# DATA304 · 저장한 체크포인트의 검색 단계별 검증

기존 Gold(epoch 2)와 추가 학습으로 저장한 epoch 3·4를 같은 VAL에서 비교한다. 이 노트북은 평가만 수행한다. `cross_gold.pt`는 세 경우 모두 같은 파일을 읽는다. 기존 Gold의 상품 인덱스는 `data304_outputs/index_gold`에서, 추가 에포크의 체크포인트·인덱스·검증 결과는 `data304_outputs/additional_training/epoch_XX/`에서 읽는다.

기존 SageMaker 학습 커널에서 **Run All**을 실행한다. 경로는 첫 셀에서 확인한다. 추가 학습 체크포인트가 아직 없으면 사용 가능한 에포크를 출력한다. 먼저 학습 노트북을 실행했다면 같은 VAL·검색 설정에서 저장된 검증 CSV가 재사용된다. FAISS가 없는 커널에서는 `%pip install faiss-cpu` 후 같은 커널을 재시작한다.

In [ ]:
from pathlib import Path
from contextlib import nullcontext
from functools import lru_cache
from itertools import islice, product
from tempfile import TemporaryDirectory
from collections import defaultdict
import copy
import hashlib
import itertools
import json
import math
import os
import random

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer

# SageMaker에서는 이전 실행의 절대경로를 쓴다. 프로젝트 폴더를 통째로 옮겼다면 이 한 줄을 바꾼다.
PROJECT_DIR = Path('/home/ec2-user/SageMaker/20262R0136DATA30400')
RUN_DIR = PROJECT_DIR / 'data304_outputs'
GOLD_RESUME = RUN_DIR / 'checkpoints' / 'colbert_gold_last.pt'
GOLD_WEIGHTS = RUN_DIR / 'checkpoints' / 'colbert_gold.pt'
CROSS_WEIGHTS = RUN_DIR / 'checkpoints' / 'cross_gold.pt'
GOLD_INDEX = RUN_DIR / 'index_gold'
SPLIT_PATH = RUN_DIR / 'split.json'
VOCAB_PATH = RUN_DIR / 'aspect_vocabulary_and_config.json'
ASPECT_PATH = RUN_DIR / 'metadata_aspects.json'
CATALOG_PATH = PROJECT_DIR / 'catalog.json'
GOLD_PATH = PROJECT_DIR / 'train_gold.json'
OUTPUT_DIR = RUN_DIR / 'additional_training'

# 이전 진단과 같은 VAL 350건을 같은 순서로 쓴다. None이면 저장한 VAL 전체를 쓴다.
INCLUDE_GOLD_BASELINE = True  # 기존 Gold 인덱스를 사용해 추가 학습 전 기준점도 평가한다.
INCLUDE_GOLD_BASELINE = False

VAL_LIMIT = 350

TOKEN_NEIGHBORS = (96,)
ANN_POOL_SIZES = (900,)
COLBERT_TOPKS = (100,)

EXTRA_EPOCHS = 2
LEARNING_RATE = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
faiss.omp_set_num_threads(min(8, os.cpu_count() or 4))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 이 세 값이 CSV의 각 행에도 그대로 들어간다. 설정 셀 실행 직후 8개 조합을 확인할 수 있다.
GRID = list(product(TOKEN_NEIGHBORS, ANN_POOL_SIZES, COLBERT_TOPKS))
GRID_COLUMNS = ['ann_per_token', 'ann_pool', 'candidate_topk']
if not GRID or any(not (n > 0 and p >= k > 0) for n, p, k in GRID) or len(GRID) != len(set(GRID)):
    raise ValueError('검색 설정에는 중복 없는 양의 정수와 ann_pool >= candidate_topk가 필요합니다.')
print('Gold 재개 파일', GOLD_RESUME, '\n고정 재랭커', CROSS_WEIGHTS, '\n결과 폴더', OUTPUT_DIR, flush=True)
display(pd.DataFrame(GRID, columns=GRID_COLUMNS))

## 저장한 학습 설정과 분할

카탈로그 행은 FAISS 상품 번호이고, 저장된 VAL 순서는 이전 350건과 비교하는 기준이다.

In [ ]:
# 학습 당시와 같은 문자열, 토큰 길이, 속성 열 순서, query_id 순서를 복원한다.
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))


def write_json(path, value):
    # 완성된 설정을 나중에 점수표와 대조할 수 있도록 사람이 읽을 수 있는 형식으로 쓴다.
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


@lru_cache(maxsize=32)
def file_hash(path, size, modified_ns):
    # 모델과 입력이 같을 때만 캐시를 공유한다. 같은 실행에서 큰 파일을 매번 다시 읽지 않는다.
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def sha256(path):
    # 파일을 교체했으면 크기 또는 수정 시각이 바뀌므로 새 해시를 얻는다.
    stat = path.stat()
    return file_hash(path, stat.st_size, stat.st_mtime_ns)


def signature(value):
    # 순서가 같은 VAL과 같은 검색 설정에 재실행 시 같은 디렉터리를 배정한다.
    data = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(data).hexdigest()


def epoch_dir(number):
    # Gold와 추가 에포크를 구별해 원래 checkpoints 및 index_gold를 보존한다.
    return OUTPUT_DIR / f'epoch_{number:02d}'


def amp_context():
    # 원래 실행처럼 지원되는 CUDA에서는 bfloat16으로 순전파한다.
    return torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()


def load_pt(path):
    # CPU 메모리 매핑으로 1 GB가 넘는 AdamW 체크포인트도 필요한 텐서부터 읽는다.
    return torch.load(path, map_location='cpu', mmap=True, weights_only=True)


def metadata_text(item):
    # 학습 때 사용한 네 필드의 이름, 순서, [SEP]를 그대로 지켜 임베딩 입력을 맞춘다.
    fields = [
        ('title', 'Product'),
        ('store', 'Store'),
        ('features', 'Features'),
        ('description', 'Description'),
    ]
    parts = [f'{label}: {item[key].strip()}' for key, label in fields if item[key].strip()]
    return ' [SEP] '.join(parts) or 'Product metadata not available'


def gold_query(row):
    # item_id가 상품 정답이고 query_id가 묶음 식별자다. 리뷰 다섯 개는 원래 순서로 남긴다.
    if len(row['reviews']) != 5:
        raise ValueError(f'리뷰가 다섯 개가 아닌 쿼리 {row["query_id"]}')
    path = tuple(part.strip() for part in row['category_path'].split('>') if part.strip())
    return dict(
        query_id=row['query_id'],
        item_id=row['item_id'],
        category_path=path,
        reviews=[r['text'].strip() for r in row['reviews']],
    )


for path in (CATALOG_PATH, GOLD_PATH, SPLIT_PATH, VOCAB_PATH, GOLD_RESUME, CROSS_WEIGHTS):
    if not path.is_file():
        raise FileNotFoundError(f'기존 실행 파일을 확인하세요. {path}')
vocab_record = read_json(VOCAB_PATH)
CFG = dict(vocab_record['config'])
manifest_path = RUN_DIR / 'run_manifest.json'
if manifest_path.is_file():
    CFG.update(read_json(manifest_path)['config'])
# FAISS IVF 검색 범위 변경
CFG['ann_nprobe'] = 128
ASPECT_KEYS = ('category', 'brand', 'color', 'material', 'size', 'features')
# AdamW가 파라미터를 순서로 연결하므로 원본에서 속성 헤드를 등록한 순서도 복원한다.
VOCABS = {key: vocab_record['vocabularies'][key] for key in ASPECT_KEYS}
catalog = read_json(CATALOG_PATH)
ITEM_IDS = [item['item_id'] for item in catalog]
ITEM_TO_IDX = {item_id: i for i, item_id in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(item) for item in catalog]
del catalog
rows = read_json(GOLD_PATH)
by_id = {row['query_id']: gold_query(row) for row in rows}
if len(by_id) != len(rows) or len(ITEM_IDS) != len(ITEM_TO_IDX):
    raise ValueError('카탈로그의 item_id 또는 gold의 query_id가 중복됩니다.')
del rows

split = read_json(SPLIT_PATH)
SEED = int(split['seed'])
train_ids, val_ids = split['train_query_ids'], split['val_query_ids']
TRAIN = [by_id[qid] for qid in train_ids]
full_val = [by_id[qid] for qid in val_ids]
if (
    not TRAIN
    or not full_val
    or len(train_ids) != len(set(train_ids))
    or len(val_ids) != len(set(val_ids))
    or {r['item_id'] for r in TRAIN} & {r['item_id'] for r in full_val}
    or any(r['item_id'] not in ITEM_TO_IDX for r in itertools.chain(TRAIN, full_val))
):
    raise ValueError('저장된 학습·검증 분할의 상품 ID와 중복 여부를 확인하세요.')
if VAL_LIMIT is not None and VAL_LIMIT < 1:
    raise ValueError('VAL_LIMIT에는 양의 정수 또는 None을 지정하세요.')
VAL = full_val[:VAL_LIMIT]
del by_id, full_val
BASE_EPOCH = int(load_pt(GOLD_RESUME)['epoch'])  # 원래 저장 코드는 완료한 횟수(epoch + 1)를 기록했다.
TARGET_EPOCHS = range(BASE_EPOCH + 1, BASE_EPOCH + EXTRA_EPOCHS + 1)
print(
    f'{DEVICE} | 기존 {BASE_EPOCH} epoch 완료 | TRAIN {len(TRAIN):,} | VAL {len(VAL):,}/{len(val_ids):,}',
    flush=True,
)

## 체크포인트와 같은 모델 구조

기존 속성 헤드·pair 편향까지 포함해 저장된 가중치와 파라미터 순서를 복원한다.

In [ ]:
# 체크포인트에는 구조 설정과 토크나이저 파일이 없다. 이전 학습과 같은 백본 이름으로 둘을 읽는다.
BACKBONE_CONFIG = AutoConfig.from_pretrained(CFG['backbone'])
TOKENIZER = AutoTokenizer.from_pretrained(CFG['backbone'], use_fast=True)
SPECIAL_IDS = set(TOKENIZER.all_special_ids)


# 텍스트 배치를 토큰 ID와 attention mask 텐서로 바꾸고, 길이 제한과 패딩을 적용해 모델의 실행 장치로 보낸다.
def tokenize(texts, max_length):
    out = TOKENIZER(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    return {k: v.to(DEVICE) for k, v in out.items()}


# 상품 설명과 리뷰를 각각 인코딩한 뒤 토큰끼리 비교하는 검색기다. 같은 상품 설명을 여러 쿼리에 재사용할 수 있다.
class ColBERTAspects(nn.Module):
    # 같은 백본으로 리뷰·상품 인코더를 따로 만들고, 각자의 토큰 투영층과 공통 속성 헤드를 준비한다.
    def __init__(self):
        super().__init__()
        self.query_encoder = AutoModel.from_config(BACKBONE_CONFIG)
        self.doc_encoder = AutoModel.from_config(BACKBONE_CONFIG)
        h = self.query_encoder.config.hidden_size
        self.query_proj = nn.Linear(h, CFG['dimension'], bias=False)
        self.doc_proj = nn.Linear(h, CFG['dimension'], bias=False)
        self.aspect_heads = nn.ModuleDict({k: nn.Linear(h, len(v)) for k, v in VOCABS.items() if len(v) > 0})
        # 속성 예측에는 공통 aspect_heads를 쓴다. 아래 Identity 모듈에는 학습할 가중치가 없다.
        self.query_to_aspects = nn.Identity()
        self.doc_to_aspects = nn.Identity()
        # 리뷰 쌍 분류 전용 학습 파라미터는 내적 점수에 더하는 스칼라 편향 b 하나다.
        # 리뷰 벡터는 기존 query_encoder와 query_proj에서 오므로 별도의 임베딩 테이블이나 MLP가 필요하지 않다.
        self.pair_bias = nn.Parameter(torch.zeros(()))
        self.pair_bce = nn.BCEWithLogitsLoss()

    # 한 번 인코딩해 랭킹·리뷰 쌍 분류용 토큰 벡터와 속성 분류용 hidden 평균을 함께 얻는다.
    # 마지막 반환값은 어떤 위치가 유효한 토큰인지를 나타낸다. 평균과 MaxSim을 계산할 때 같은 위치를 선택한다.
    def encode(self, tokens, side):
        backbone = self.query_encoder if side == 'query' else self.doc_encoder
        projection = self.query_proj if side == 'query' else self.doc_proj
        hidden = backbone(**tokens).last_hidden_state
        # 패딩과 특수 토큰 위치를 표시해 이후 평균과 MaxSim에서 사용할 본문 토큰을 정한다.
        mask = tokens['attention_mask'].bool()
        special = torch.zeros_like(mask)
        for t in SPECIAL_IDS:
            special |= tokens['input_ids'].eq(t)
        valid = mask & ~special
        # 본문 토큰이 없으면 attention mask를 써서 [CLS]·[SEP]처럼 패딩이 아닌 토큰을 살린다.
        empty = ~valid.any(dim=-1)
        if empty.any():
            valid[empty] = mask[empty]
        # 속성 분류에는 투영 전 hidden 평균을, 랭킹과 리뷰 쌍 분류에는 투영·정규화한 토큰 벡터를 사용한다.
        pooled = (hidden * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp(min=1)
        projected = F.normalize(projection(hidden), p=2, dim=-1)
        return projected, pooled, valid

    # MaxSim은 각 리뷰 토큰이 후보 상품에서 가장 잘 맞는 토큰 하나를 골라 그 내적을 점수로 쓰는 연산이다.
    # 이 값들을 리뷰 안에서 합하고, 다섯 리뷰의 점수를 평균내 각 상품 후보의 점수를 구한다.
    def scores(self, q_emb, q_mask, d_emb, d_mask):
        # q_emb는 [배치, 리뷰 5개, 리뷰 토큰, 차원], d_emb는 [배치, 후보 상품, 상품 토큰, 차원]이다.
        dots = torch.einsum('brqd,bcmd->brcqm', q_emb, d_emb)
        dots = dots.masked_fill(~d_mask[:, None, :, None, :], -1e4)
        maxima = dots.max(-1).values
        maxima = maxima * q_mask[:, :, None, :]
        return maxima.sum(-1).mean(1)  # 출력 크기 [배치, 후보 상품 수]


# 재랭커와 baseline은 리뷰 다섯 개를 한 텍스트로 읽는다. 각 리뷰를 자른 뒤 번호와 [SEP]로 경계를 표시한다.
def joined_reviews(reviews):
    # 리뷰별 상한을 먼저 적용한다. 재랭커가 상품과 합쳐 다시 자를 때 뒤쪽 리뷰가 더 짧아질 수 있다.
    chunks = []
    for i, r in enumerate(reviews):
        ids = TOKENIZER.encode(
            r, add_special_tokens=False, truncation=True, max_length=CFG['cross_tokens_per_review']
        )
        chunks.append(f'Review {i + 1}: ' + TOKENIZER.decode(ids, skip_special_tokens=True))
    return ' [SEP] '.join(chunks)


# 검색기가 추린 후보마다 다섯 리뷰와 해당 상품 설명을 함께 읽고, 관련도 점수 하나를 내는 재랭커다.
class CrossReranker(nn.Module):
    # 저장된 cross_gold.pt의 구조를 만든다. 가중치는 아래 load_cross에서 엄격히 복원한다.
    def __init__(self):
        super().__init__()
        config = copy.deepcopy(BACKBONE_CONFIG)
        config.num_labels = 1
        self.net = AutoModelForSequenceClassification.from_config(config)

    # 각 리뷰 묶음과 후보 메타데이터를 한 쌍으로 인코딩해, 쌍마다 관련도 점수(logit) 하나를 반환한다.
    def score(self, reviews, doc_texts):
        # 총길이가 cross_tokens를 넘으면 더 긴 쪽부터 줄인다. 이때 뒤쪽 리뷰 전체가 잘릴 수도 있다.
        inputs = TOKENIZER(
            reviews,
            doc_texts,
            padding=True,
            truncation='longest_first',
            max_length=CFG['cross_tokens'],
            return_tensors='pt',
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        return self.net(**inputs).logits.reshape(-1)


def load_cross():
    # 모든 에포크가 같은 Gold 재랭커를 사용한다. dropout을 끄고 gradient 계산 대상에서도 제외한다.
    model = CrossReranker().to(DEVICE)
    model.load_state_dict(load_pt(CROSS_WEIGHTS), strict=True)
    return model.eval().requires_grad_(False)

## 검색 설정과 지표

세 설정을 독립적으로 바꾼다. 각 리뷰 토큰의 FAISS 이웃 수는 **48·96**, ANN 점수로 고르는 상품 수는 **450·900**, MaxSim으로 골라 재랭커에 넘기는 상품 수는 **50·100**이다. `ann_nprobe`는 기존 실행 설정을 따른다.

| 지표 | 쿼리 하나에서 적중으로 세는 조건 |
|---|---|
| ANN Recall@450 또는 @900 | 보충 전 ANN 상위 450개 또는 900개 상품에 정답이 있다. |
| ColBERT Recall@50 또는 @100 | 해당 풀의 exact MaxSim 상위 50개 또는 100개에 정답이 있다. |
| Cross Precision@1 | 고정된 재랭커가 후보 중 1위로 고른 상품이 정답이다. |
| Cross Recall@10 | 이전 진단의 0.39와 같은 지표로, 재랭커 상위 10개에 정답이 있다. |

모든 비율의 분모는 같은 VAL 쿼리 수다. 앞 단계에서 정답을 놓친 쿼리도 최종 지표의 분모에 포함한다. 각 에포크의 `validation/<설정 해시>/`에 8행 요약인 `stage_summary.csv`, 쿼리별 순위와 적중 여부인 `query_diagnostics.csv`, 실제 입력과 설정인 `settings.json`을 저장한다.

In [ ]:
# 이 에포크의 상품 인코더로 전체 카탈로그를 한 번 인코딩하고, 유효 토큰으로 FAISS IVF를 만든다.
@torch.inference_mode()
def build_token_index(model, folder):
    folder.mkdir(parents=True)
    n_items, length, dim = len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension']
    emb = np.lib.format.open_memmap(
        folder / 'doc_embeddings.npy', mode='w+', dtype=np.float16, shape=(n_items, length, dim)
    )
    mask = np.lib.format.open_memmap(
        folder / 'doc_masks.npy', mode='w+', dtype=np.bool_, shape=(n_items, length)
    )
    batch = CFG['index_doc_batch']
    for start in tqdm(range(0, n_items, batch), desc='Encode catalog'):
        end = min(n_items, start + batch)
        tokens = tokenize(CAT_TEXTS[start:end], length)
        with amp_context():
            reps, _, valid = model.encode(tokens, 'doc')
        width = reps.shape[1]
        emb[start:end] = 0
        mask[start:end] = False
        emb[start:end, :width] = reps.float().cpu().numpy().astype(np.float16)
        mask[start:end, :width] = valid.cpu().numpy()
    emb.flush()
    mask.flush()

    # 학습 노트북과 같은 방식으로 최대 12만 유효 토큰을 골라 IVF의 군집을 구성한다.
    valid_docs, valid_positions = np.where(mask)
    n_train = min(120_000, len(valid_docs))
    sample = np.random.default_rng(SEED).choice(len(valid_docs), size=n_train, replace=False)
    train_vectors = np.ascontiguousarray(emb[valid_docs[sample], valid_positions[sample]], dtype=np.float32)
    token2doc = valid_docs.astype(np.int32)
    del valid_docs, valid_positions
    nlist = min(CFG['ann_nlist'], max(1, n_train // 40))
    index = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, nlist, faiss.METRIC_INNER_PRODUCT)
    index.train(train_vectors)
    index.nprobe = min(CFG['ann_nprobe'], nlist)
    batch = CFG['index_add_docs']
    for start in tqdm(range(0, n_items, batch), desc='Build token ANN'):
        end = min(n_items, start + batch)
        valid = mask[start:end].reshape(-1)
        vectors = emb[start:end].reshape(-1, dim)[valid]
        if len(vectors):
            index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    assert index.ntotal == len(token2doc)
    faiss.write_index(index, str(folder / 'token_ivf.index'))
    np.save(folder / 'token_to_item.npy', token2doc)


def open_token_index(folder):
    # 같은 인덱스의 네 파일을 읽고 상품 수·토큰 길이·임베딩 차원이 맞는지 확인한다.
    emb = np.load(folder / 'doc_embeddings.npy', mmap_mode='r')
    mask = np.load(folder / 'doc_masks.npy', mmap_mode='r')
    token2doc = np.load(folder / 'token_to_item.npy', mmap_mode='r')
    index = faiss.read_index(str(folder / 'token_ivf.index'))
    expected = (len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension'])
    if (
        emb.shape != expected
        or mask.shape != expected[:2]
        or mask.dtype != np.bool_
        or index.d != expected[2]
        or token2doc.ndim != 1
        or index.ntotal != len(token2doc)
    ):
        raise ValueError(f'상품 인덱스의 크기가 현재 설정과 맞지 않습니다. {folder}')
    if not len(token2doc) or token2doc.min() < 0 or token2doc.max() >= len(ITEM_IDS):
        raise ValueError(f'인덱스의 토큰-상품 대응표를 확인하세요. {folder}')
    index.nprobe = min(CFG['ann_nprobe'], index.nlist)
    return {'index': index, 'token2doc': token2doc, 'emb': emb, 'mask': mask}


def saved_gold_index(number):
    # 기존 Gold 인덱스의 네 파일이 모두 준비됐는지 확인하고, 파일 교체 여부를 평가 캐시에 반영한다.
    if number != BASE_EPOCH:
        return None
    files = [
        GOLD_INDEX / name
        for name in ('token_ivf.index', 'token_to_item.npy', 'doc_embeddings.npy', 'doc_masks.npy')
    ]
    if not all(path.is_file() for path in files):
        return None
    return {path.name: (path.stat().st_size, path.stat().st_mtime_ns) for path in files}


def index_for_epoch(model, number, weights_path):
    # 추가 학습에서는 상품 인코더도 바뀐다. 체크포인트별 상품 벡터와 IVF를 한 번만 만든다.
    model.eval()
    spec = {
        'weights': sha256(weights_path),
        'catalog': sha256(CATALOG_PATH),
        'seed': SEED,
        'precision': 'bfloat16' if USE_BF16 else 'float32',
        'config': {
            k: CFG[k]
            for k in (
                'backbone',
                'dimension',
                'metadata_tokens',
                'ann_nlist',
                'index_doc_batch',
                'index_add_docs',
            )
        },
    }
    cache = epoch_dir(number) / 'indexes'
    folder = cache / signature(spec)[:20]
    if saved_gold_index(number) is not None:
        # 이 폴더는 같은 Gold 실행에서 만든 기존 인덱스다. epoch 3·4는 각자 만든 인덱스만 연다.
        folder = GOLD_INDEX
    elif not folder.is_dir():
        cache.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=cache) as tmp:
            staged = Path(tmp) / 'index'
            print('상품 인덱스 생성', folder, flush=True)
            build_token_index(model, staged)
            write_json(staged / 'index_info.json', spec)
            publish_directory(staged, folder)
    print('검증 인덱스', folder, flush=True)
    return open_token_index(folder), folder


def publish_directory(staged, destination):
    # 같은 파일시스템에서 완성된 폴더만 공개한다. 동시 실행에서 먼저 완성된 캐시를 공유한다.
    try:
        staged.rename(destination)
    except OSError:
        if not destination.is_dir():
            raise

In [ ]:
# 리뷰 다섯 개를 각각 인코딩해 토큰 벡터와 마스크를 NumPy 배열로 반환한다. 리뷰 구분은 그대로 남긴다.
@torch.inference_mode()
def encode_five_reviews(model, revs):
    model.eval()
    assert len(revs) == 5
    tok = tokenize(revs, CFG['query_tokens'])
    with amp_context():
        rep, _, valid = model.encode(tok, 'query')
    return rep.float().cpu().numpy(), valid.cpu().numpy()


def token_hit_order(query, query_mask, index_bundle, per_token):
    # 동일 리뷰 토큰이 한 상품의 여러 토큰을 찾으면 그중 가장 큰 내적만 사용한다.
    token_vectors = np.ascontiguousarray(query[query_mask], dtype=np.float32)
    distances, token_ids = index_bundle['index'].search(token_vectors, per_token)
    token2doc = index_bundle['token2doc']
    rough = defaultdict(float)
    for similarities, hits in zip(distances, token_ids):
        best = {}
        for similarity, token_id in zip(similarities, hits):
            if token_id < 0:
                continue
            doc = int(token2doc[token_id])
            best[doc] = max(best.get(doc, -1e6), float(similarity))
        for doc, similarity in best.items():
            rough[doc] += max(0.0, similarity)
    return sorted(rough, key=lambda doc: (-rough[doc], doc))


@torch.inference_mode()
def maxsim_scores(query, query_mask, index_bundle, doc_indices):
    # 후보의 유효 상품 토큰 전체를 읽고, 리뷰 토큰마다 가장 높은 내적을 더한다.
    # 최종 점수 하나는 다섯 리뷰의 MaxSim 점수 평균이다.
    qe = torch.as_tensor(query, device=DEVICE)
    qm = torch.as_tensor(query_mask, device=DEVICE)
    scores = []
    for start in range(0, len(doc_indices), 96):
        rows = doc_indices[start : start + 96]
        de = torch.as_tensor(np.asarray(index_bundle['emb'][rows]).astype(np.float32), device=DEVICE)
        dm = torch.as_tensor(np.asarray(index_bundle['mask'][rows]), device=DEVICE)
        with amp_context():
            simil = torch.einsum('rqd,cmd->rcqm', qe, de)
            simil = simil.masked_fill(~dm[None, :, None, :], -1e4)
            review_scores = (simil.max(-1).values * qm[:, None, :]).sum(-1)
        scores.append(review_scores.mean(0).float().cpu().numpy())
    return np.concatenate(scores)


# 후보 상품별로 cross-encoder 점수를 배치 계산하고, 점수가 높은 순서대로 상품 ID와 점수를 함께 반환한다.
@torch.inference_mode()
def rerank_candidate_ids(model, reviews, ids):
    model.eval()
    q = joined_reviews(reviews)
    result = []
    for start in range(0, len(ids), CFG['cross_infer_batch']):
        sub = ids[start : start + CFG['cross_infer_batch']]
        with amp_context():
            logits = model.score([q] * len(sub), [CAT_TEXTS[ITEM_TO_IDX[iid]] for iid in sub])
        result.extend(logits.float().cpu().numpy().tolist())
    order = np.argsort(-np.asarray(result), kind='stable')
    return [ids[i] for i in order], [result[i] for i in order]

In [ ]:
def scored_pool(ann_order, pool_size, topk):
    # ANN 적중 여부는 보충 전 순위로 센다. 실제 후보 부족 시에만 원래 검색기와 같은 순서로 보충한다.
    pool = ann_order[:pool_size]
    if len(pool) < topk:
        seen = set(pool)
        need = min(pool_size, len(ITEM_IDS)) - len(pool)
        pool.extend(islice((i for i in range(len(ITEM_IDS)) if i not in seen), need))
    return pool


@torch.inference_mode()
def evaluate_query(row, model, cross, bundle):
    # 한 쿼리를 한 번 인코딩하고, 48개 이웃 검색과 96개 이웃 검색은 각각 실제 FAISS로 실행한다.
    query, mask = encode_five_reviews(model, row['reviews'])
    gold = ITEM_TO_IDX[row['item_id']]
    neighbor_counts = dict.fromkeys(n for n, _, _ in GRID)
    ann_orders = {n: token_hit_order(query, mask, bundle, n) for n in neighbor_counts}
    ann_ranks = {n: order.index(gold) + 1 if gold in order else None for n, order in ann_orders.items()}
    pools = {(n, p, k): scored_pool(ann_orders[n], p, k) for n, p, k in GRID}

    # 같은 상품의 MaxSim은 모든 조합에서 같다. 합집합을 채점한 뒤 각 조합의 자기 풀 안에서 다시 정렬한다.
    all_docs = list(dict.fromkeys(i for pool in pools.values() for i in pool))
    scores = dict(zip(all_docs, maxsim_scores(query, mask, bundle, all_docs)))
    rankings, candidates = {}, {}
    for setting, pool in pools.items():
        rankings[setting] = sorted(pool, key=scores.__getitem__, reverse=True)
        candidates[setting] = [ITEM_IDS[i] for i in rankings[setting][: setting[2]]]

    # cross 점수도 (리뷰 묶음, 상품)마다 하나다. 동점이면 해당 조합의 MaxSim 순서를 유지한다.
    all_candidates = list(dict.fromkeys(iid for group in candidates.values() for iid in group))
    cross_ids, logits = rerank_candidate_ids(cross, row['reviews'], all_candidates)
    cross_scores = dict(zip(cross_ids, logits))
    records = []
    for setting, ids in candidates.items():
        n, pool_size, topk = setting
        exact_order = rankings[setting]
        final = sorted(ids, key=cross_scores.__getitem__, reverse=True)
        ann_rank = ann_ranks[n]
        maxsim_rank = exact_order.index(gold) + 1 if gold in exact_order else None
        cross_rank = final.index(row['item_id']) + 1 if row['item_id'] in final else None
        records.append(
            {
                'query_id': row['query_id'],
                'gold_item_id': row['item_id'],
                'ann_per_token': n,
                'ann_pool': pool_size,
                'candidate_topk': topk,
                'gold_ann_rank': ann_rank,
                'gold_maxsim_rank_in_pool': maxsim_rank,
                'cross_gold_rank': cross_rank,
                'ann_hit': ann_rank is not None and ann_rank <= pool_size,
                'colbert_hit': maxsim_rank is not None and maxsim_rank <= topk,
                'cross_top1_item_id': final[0],
                'cross_top1_hit': row['item_id'] == final[0],
                'cross_top10_hit': cross_rank is not None and cross_rank <= 10,
            }
        )
    return records


def evaluate_epoch(number, weights_path, model, cross):
    # 검증 입력·체크포인트·검색 조건을 기록한다. 확인된 동일 평가가 있으면 저장한 CSV를 바로 읽는다.
    spec = {
        'evaluation_version': 1,
        'epoch': number,
        'additional_epoch': number - BASE_EPOCH,
        'colbert_checkpoint': str(weights_path),
        'colbert_sha256': sha256(weights_path),
        'cross_checkpoint': str(CROSS_WEIGHTS),
        'cross_sha256': sha256(CROSS_WEIGHTS),
        'split': str(SPLIT_PATH),
        'split_sha256': sha256(SPLIT_PATH),
        'gold_sha256': sha256(GOLD_PATH),
        'catalog_sha256': sha256(CATALOG_PATH),
        'original_index_files': saved_gold_index(number),
        'n_queries': len(VAL),
        'val_query_ids': [row['query_id'] for row in VAL],
        'search_settings': [dict(zip(GRID_COLUMNS, setting)) for setting in GRID],
        'precision': 'bfloat16' if USE_BF16 else 'float32',
        'seed': SEED,
        'config': {
            k: CFG[k]
            for k in (
                'backbone',
                'dimension',
                'query_tokens',
                'metadata_tokens',
                'cross_tokens',
                'cross_tokens_per_review',
                'cross_infer_batch',
                'ann_nlist',
                'ann_nprobe',
                'index_doc_batch',
                'index_add_docs',
            )
        },
    }
    parent = epoch_dir(number) / 'validation'
    report = parent / signature(spec)[:20]
    if not (report / 'stage_summary.csv').is_file():
        bundle, index_path = index_for_epoch(model, number, weights_path)
        model.eval()
        cross.eval()
        records = []
        for row in tqdm(VAL, desc=f'VAL epoch {number} · {len(GRID)}개 설정'):
            records.extend(evaluate_query(row, model, cross, bundle))
        detail = pd.DataFrame.from_records(records)
        detail.insert(0, 'epoch', number)
        summary = detail.groupby(GRID_COLUMNS, sort=False, as_index=False).agg(
            n_queries=('query_id', 'size'),
            ann_hits=('ann_hit', 'sum'),
            colbert_hits=('colbert_hit', 'sum'),
            top1_hits=('cross_top1_hit', 'sum'),
            ann_recall_at_pool=('ann_hit', 'mean'),
            colbert_recall_at_k=('colbert_hit', 'mean'),
            cross_precision_at_1=('cross_top1_hit', 'mean'),
            cross_recall_at_10=('cross_top10_hit', 'mean'),
        )
        summary.insert(0, 'epoch', number)
        summary.insert(1, 'additional_epoch', number - BASE_EPOCH)
        # 고정된 이름의 지표는 실제로 그 개수를 쓴 행에만 채운다. 450·50 실험은 앞의 가변 K 열에서 읽는다.
        summary['ann_recall_at_900'] = summary['ann_recall_at_pool'].where(summary['ann_pool'].eq(900))
        summary['colbert_recall_at_100'] = summary['colbert_recall_at_k'].where(
            summary['candidate_topk'].eq(100)
        )
        parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=parent) as tmp:
            staged = Path(tmp) / 'report'
            staged.mkdir()
            detail.to_csv(staged / 'query_diagnostics.csv', index=False)
            summary.to_csv(staged / 'stage_summary.csv', index=False)
            write_json(staged / 'settings.json', {**spec, 'index_path': str(index_path)})
            publish_directory(staged, report)
    print('검증 결과', report, flush=True)
    return pd.read_csv(report / 'stage_summary.csv').assign(report_dir=str(report))


def show_comparison(parts, filename):
    # 에포크·검색폭별 결과를 한 파일에 모은다. 두 번째 표는 요청한 900 → 100 → 1 비교에 해당한다.
    combined = pd.concat(parts, ignore_index=True).sort_values(['epoch', *GRID_COLUMNS])
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=OUTPUT_DIR) as tmp:
        local = Path(tmp) / filename
        combined.to_csv(local, index=False)
        local.replace(OUTPUT_DIR / filename)
    print('전체 조합', OUTPUT_DIR / filename, flush=True)
    display(
        combined[
            [
                'epoch',
                *GRID_COLUMNS,
                'n_queries',
                'ann_recall_at_pool',
                'colbert_recall_at_k',
                'cross_precision_at_1',
                'cross_recall_at_10',
            ]
        ].round(4)
    )
    focus = combined.loc[combined['ann_pool'].eq(900) & combined['candidate_topk'].eq(100)]
    if not focus.empty:
        print('ANN 900 → ColBERT 100 → Cross 1')
        display(
            focus[
                [
                    'epoch',
                    'additional_epoch',
                    'ann_per_token',
                    'n_queries',
                    'ann_recall_at_900',
                    'colbert_recall_at_100',
                    'cross_precision_at_1',
                ]
            ].round(4)
        )
    return combined

## 실행과 결과

에포크별 CSV에는 실제 검색 설정과 쿼리 수가 함께 기록된다. 마지막 표에서 두 이웃 수에 대한 Recall@900, Recall@100, Precision@1을 바로 비교할 수 있다.

In [ ]:
def compare_saved_epochs():
    # 원래 Gold와 디스크에 저장이 끝난 추가 에포크를 같은 split, 토크나이저, Cross로 비교한다.
    model = ColBERTAspects().to(DEVICE)
    cross = load_cross()
    parts = []
    numbers = ([BASE_EPOCH] if INCLUDE_GOLD_BASELINE else []) + list(TARGET_EPOCHS)
    for number in numbers:
        if number == BASE_EPOCH:
            weights = GOLD_WEIGHTS
            if not weights.is_file():
                raise FileNotFoundError(f'기존 Gold 검색기 가중치가 필요합니다. {weights}')
        else:
            weights = epoch_dir(number) / 'checkpoints' / 'colbert_gold.pt'
            if not weights.is_file():
                print(f'추가 {number - BASE_EPOCH} epoch 가중치가 아직 없습니다. {weights}', flush=True)
                continue
        model.load_state_dict(load_pt(weights), strict=True)
        parts.append(evaluate_epoch(number, weights, model.eval(), cross))
    if parts:
        return show_comparison(parts, 'diagnostics_summary.csv')
    print('추가 학습 노트북에서 첫 에포크의 체크포인트를 저장하면 비교할 수 있습니다.')
    return pd.DataFrame()


results = compare_saved_epochs()